# Módulo 04 · Clasificación y pruebas de hipótesis

**Caso 1 – Diagnóstico:** clasificar tumores de mama como malignos o benignos a partir de 30 medidas de imágenes celulares. Un falso negativo (no detectar un cáncer) cuesta mucho más que un falso positivo (una biopsia innecesaria).

**Caso 2 – Prueba A/B:** una tienda online prueba un nuevo botón de compra. ¿La mejora observada es real o fruto del azar?

Objetivos:
1. Entrenar y comparar regresión logística, árbol de decisión, random forest y gradient boosting.
2. Elegir métricas según el coste de los errores; ajustar el **umbral de decisión**.
3. Plantear y resolver una prueba de hipótesis; entender p-valor, potencia y tamaño muestral.
4. Evitar el error de las comparaciones múltiples.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             roc_auc_score, RocCurveDisplay, precision_recall_curve)
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="whitegrid")

# Parte 1 · Clasificación
Codificamos **maligno = 1** (la clase que queremos detectar).

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = 1 - data.target  # en el dataset original 0 = maligno
print(f"{len(y)} casos | {y.mean():.0%} malignos")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

## 1. Un árbol de decisión: el modelo más fácil de explicar

In [ ]:
arbol = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
plt.figure(figsize=(16, 6))
plot_tree(arbol, feature_names=X.columns, class_names=["benigno", "maligno"], filled=True, fontsize=8)
plt.show()

## 2. Comparación de modelos

In [ ]:
modelos = {
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Árbol (prof. 3)": DecisionTreeClassifier(max_depth=3, random_state=0),
    "Random forest": RandomForestClassifier(300, random_state=0),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
fig, ax = plt.subplots(figsize=(6, 5))
filas = []
for nombre, m in modelos.items():
    m.fit(X_train, y_train)
    prob = m.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    filas.append({"modelo": nombre, "exactitud": (tp + tn) / len(y_test), "recall (sensib.)": tp / (tp + fn),
                  "precisión": tp / (tp + fp), "AUC": roc_auc_score(y_test, prob), "falsos negativos": fn})
    RocCurveDisplay.from_predictions(y_test, prob, name=nombre, ax=ax)
ax.set_title("Curvas ROC (test)"); plt.show()
pd.DataFrame(filas).set_index("modelo").round(3)

**Métricas clave**
- **Recall / sensibilidad**: de los malignos, ¿cuántos detecto? (lo que más importa aquí).
- **Precisión**: de los que marco como malignos, ¿cuántos lo son?
- **AUC**: capacidad de ordenar casos de mayor a menor riesgo, independiente del umbral.

## 3. Elegir el umbral según el coste
Supongamos que un falso negativo cuesta **20 veces** más que un falso positivo. El umbral 0,5 no tiene por qué ser el óptimo. Para no "hacer trampa" con el test, elegimos el umbral con predicciones de validación cruzada sobre el train.

In [ ]:
logit = modelos["Regresión logística"]
prob_cv = cross_val_predict(logit, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]

COSTE_FN, COSTE_FP = 20, 1
umbrales = np.linspace(0.01, 0.99, 99)
costes = [COSTE_FN * ((prob_cv < u) & (y_train == 1)).sum() + COSTE_FP * ((prob_cv >= u) & (y_train == 0)).sum() for u in umbrales]
u_opt = umbrales[np.argmin(costes)]

plt.plot(umbrales, costes); plt.axvline(u_opt, c="r", ls="--", label=f"óptimo = {u_opt:.2f}")
plt.xlabel("umbral"); plt.ylabel("coste total (CV)"); plt.legend(); plt.show()

In [ ]:
prob_test = logit.predict_proba(X_test)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, u in zip(axes, [0.5, u_opt]):
    ConfusionMatrixDisplay.from_predictions(y_test, (prob_test >= u).astype(int), display_labels=["benigno", "maligno"], ax=ax, colorbar=False)
    ax.set_title(f"Umbral = {u:.2f}")
plt.tight_layout(); plt.show()

Bajar el umbral reduce los falsos negativos a cambio de más falsos positivos: es una **decisión de negocio**, no solo técnica.

## 4. ¿Qué variables pesan más?

In [ ]:
importancias = pd.Series(modelos["Random forest"].feature_importances_, index=X.columns).nlargest(10)
importancias.sort_values().plot.barh(); plt.title("Top 10 variables (random forest)"); plt.show()

# Parte 2 · Pruebas de hipótesis y A/B testing

## 5. Planificar antes de lanzar: ¿cuántos usuarios necesito?
Conversión actual: 10 %. Queremos detectar una mejora a 11 % (+1 punto) con nivel de significación α = 0,05 y potencia del 80 %.

In [ ]:
efecto = proportion_effectsize(0.11, 0.10)
n_por_grupo = NormalIndPower().solve_power(effect_size=efecto, alpha=0.05, power=0.8, alternative="two-sided")
print(f"Usuarios necesarios por grupo: {np.ceil(n_por_grupo):,.0f}")

## 6. Analizar el experimento
- **H₀:** la conversión de A y B es la misma.
- **H₁:** son distintas.

In [ ]:
rng = np.random.default_rng(3)
n = 15_000
conv_A = rng.binomial(1, 0.100, n)
conv_B = rng.binomial(1, 0.112, n)

exitos, ensayos = np.array([conv_B.sum(), conv_A.sum()]), np.array([n, n])
z, p = proportions_ztest(exitos, ensayos)
tasa_A, tasa_B = conv_A.mean(), conv_B.mean()
print(f"Conversión A: {tasa_A:.2%} | B: {tasa_B:.2%} | diferencia: {tasa_B - tasa_A:+.2%}")
print(f"z = {z:.2f}, p-valor = {p:.2g}")
for nombre, x in [("A", conv_A), ("B", conv_B)]:
    lo, hi = proportion_confint(x.sum(), n, method="wilson")
    print(f"  IC 95 % {nombre}: [{lo:.2%}, {hi:.2%}]")

**Interpretación correcta del p-valor:** es la probabilidad de observar una diferencia al menos tan grande **si H₀ fuera cierta**. *No* es la probabilidad de que B sea mejor.

## 7. El peligro de mirar muchas métricas (o muchos segmentos)
Simulamos 1.000 experimentos en los que se comparan **20 métricas sin ningún efecto real**. ¿En cuántos experimentos aparece al menos una métrica "significativa"?

In [ ]:
n_exp, n_metricas = 1000, 20
a = rng.normal(0, 1, (n_exp, n_metricas, 500))
b = rng.normal(0, 1, (n_exp, n_metricas, 500))
p_valores = stats.ttest_ind(a, b, axis=2).pvalue          # matriz 1000 × 20

sin_corregir = (p_valores < 0.05).any(axis=1).mean()
con_holm = np.mean([multipletests(fila, alpha=0.05, method="holm")[0].any() for fila in p_valores])
print(f"Experimentos con algún falso positivo (sin corregir): {sin_corregir:.0%}")
print(f"Experimentos con algún falso positivo (Holm):         {con_holm:.0%}")

Con 20 pruebas al 5 %, la probabilidad de al menos un falso positivo es $1-0{,}95^{20} \approx 64\,\%$. Define **una métrica principal** antes del experimento y corrige si analizas varias.

## 8. Mirar a mitad del experimento (*peeking*)
Si consultas el p-valor cada día y paras en cuanto baja de 0,05, la tasa de falsos positivos se dispara:

In [ ]:
def experimento_con_peeking(dias=20, usuarios_dia=500):
    a = rng.binomial(1, 0.1, dias * usuarios_dia)
    b = rng.binomial(1, 0.1, dias * usuarios_dia)  # sin efecto real
    for d in range(1, dias + 1):
        k = d * usuarios_dia
        if proportions_ztest([a[:k].sum(), b[:k].sum()], [k, k])[1] < 0.05:
            return True
    return False

tasa_fp = np.mean([experimento_con_peeking() for _ in range(500)])
print(f"Falsos positivos mirando cada día: {tasa_fp:.0%} (debería ser 5 %)")

## Ejercicios

1. Repite el análisis de umbral con un coste FN/FP de 5 y de 50. ¿Cómo cambia el umbral óptimo? Explica el resultado a un gerente de hospital en 3 frases.
2. Entrena un random forest con solo las 5 variables más importantes. ¿Cuánto AUC pierdes? ¿Merece la pena la simplicidad?
3. Recalcula el tamaño muestral para detectar una mejora de 10 % → 10,5 %. ¿Cuántos días necesitarías con 3.000 visitas diarias?
4. En el experimento A/B, calcula el IC 95 % de la **diferencia** de conversiones y la mejora relativa. ¿Recomendarías lanzar B? ¿Qué más querrías saber?

In [ ]:
# TU CÓDIGO AQUÍ